In [2]:
import json
import pandas as pd
import numpy as np

with open("../data/games.json") as f:
    data = json.load(f)

df = pd.DataFrame.from_dict(data, orient='index')
df = df.reset_index(drop=True)

The first thing for pre-processing is cleaning up the estimated owners feature, which will be the target variable for my model. The dataset returns these in ranges of values, as the actual numerical data is inaccesible. As such, I will represent each bucket as their midpoint to simplify the usage of the buckets going forward.

In [3]:
df['estimated_owners_raw'] = df['estimated_owners']

def parse_owners(val):
    try:
        low, high = val.split(' - ')
        return (int(low) + int(high)) / 2
    except:
        return None

df['owners_numeric'] = df['estimated_owners_raw'].apply(parse_owners)


I also need to convert tags to just a list of the names, as the data put tags into dicts with numeric values attached that don't have any value to my models. Tag similarity is important to my final product, and for that I only want the names.

In [4]:
def clean_tags(x):
    if isinstance(x, dict):
        return list(x.keys())
    elif isinstance(x, list):
        return x
    else:
        return []

df['tags_clean'] = df['tags'].apply(clean_tags)
df['num_tags'] = df['tags_clean'].apply(len)
df['has_tags'] = (df['num_tags'] > 0).astype(int)

Convert written release dates to datetime

In [5]:
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
df['year'] = df['release_date'].dt.year

Add qualifier for if game is free

In [6]:
df['is_free'] = (df['price'] == 0).astype(int)

Create New DF with dropped rows missing important values

In [7]:
df['description'] = df['detailed_description'].fillna('').astype(str)

missing_owners = df['estimated_owners_raw'].isnull().sum()
missing_desc = df['description'].isnull().sum()
empty_desc = (df['description'].str.strip() == '').sum()
empty_tags = (df['num_tags'] == 0).sum()

print(f"Missing owners: {missing_owners}")
print(f"Missing descriptions: {missing_desc}")
print(f"Empty descriptions: {empty_desc}")
print(f"Empty tags: {empty_tags}")

df = df[
    (df['description'].str.strip() != '') &
    (df['estimated_owners_raw'].notnull())
]

total = len(df)

print(f"Total rows: {total}")

Missing owners: 0
Missing descriptions: 0
Empty descriptions: 8420
Empty tags: 39265
Total rows: 114191


To properly create my target value, I need to order the defferent ranges of estimated owners and observe the distribution, then decide how I will classify each instance, likely in four categories, Low Sale, Medium Sale, High Sale, and Extreme Success.

In [17]:
buckets = [
    "0 - 20000",
    "20000 - 50000",
    "50000 - 100000",
    "100000 - 200000",
    "200000 - 500000",
    "500000 - 1000000",
    "1000000 - 2000000",
    "2000000 - 5000000",
    "5000000 - 10000000",
    "10000000 - 20000000",
    "20000000 - 50000000",
    "50000000 - 100000000",
    "100000000 - 200000000"
]

counts = df['estimated_owners_raw'].value_counts().reindex(buckets)
print(counts)

hit_threshold = [
    "50000 - 100000",
    "100000 - 200000",
    "200000 - 500000",
    "500000 - 1000000",
    "1000000 - 2000000",
    "2000000 - 5000000",
    "5000000 - 10000000",
    "10000000 - 20000000",
    "20000000 - 50000000",
    "50000000 - 100000000",
    "100000000 - 200000000"
]

df['hit'] = df['estimated_owners_raw'].isin(hit_threshold).astype(int)

print(f"Dataset shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
print("Class distribution:")
print(df['hit'].value_counts(normalize=True).rename({0: 'Miss (< 50K owners)', 1: 'Hit (≥ 50K owners)'}).map('{:.1%}'.format))

estimated_owners_raw
0 - 20000                75319
20000 - 50000            11381
50000 - 100000            5345
100000 - 200000           3449
200000 - 500000           2847
500000 - 1000000          1154
1000000 - 2000000          728
2000000 - 5000000          401
5000000 - 10000000         124
10000000 - 20000000         51
20000000 - 50000000         30
50000000 - 100000000         9
100000000 - 200000000        4
Name: count, dtype: int64
Dataset shape: 114,034 rows, 84 columns
Class distribution:
hit
Miss (< 50K owners)    87.6%
Hit (≥ 50K owners)     12.4%
Name: proportion, dtype: str


Feature Engineering: Developer Previous Success and Experience
I will be sorting the data by release date and iterating through it to have features that acknowledge the developer's previous success and experience, as these are extremely crucial features to game success as they severely impact outreach and marketing for a game. Leaving them out would make any model of this type useless.

In [9]:
df = df.sort_values('release_date').reset_index(drop=True)

In [10]:
from collections import defaultdict

dev_history = defaultdict(list)
dev_success_list = []

dev_experience_list = []

for _, row in df.iterrows():
    devs = row['developers']
    
    scores = []
    counts = []
    
    for dev in devs:
        past = dev_history[dev]
        
        counts.append(len(past))
        
        if len(past) == 0:
            scores.append(0)
        else:
            scores.append(sum(past) / len(past))
    
    dev_success = sum(scores) / len(scores) if scores else 0
    dev_experience = sum(counts) / len(counts) if counts else 0
    
    dev_success_list.append(dev_success)
    dev_experience_list.append(dev_experience)
    
    for dev in devs:
        dev_history[dev].append(row['hit'])


df['dev_success'] = dev_success_list
df['dev_experience'] = dev_experience_list
df['dev_had_success'] = (df['dev_success'] > 0).astype(int)
df['log_dev_experience'] = np.log1p(df['dev_experience'])
df['num_devs'] = df['developers'].apply(len)


In [11]:
df = df[df['num_devs'] > 0]

Now we need to encode the genre lists into numerical values for use in the models:

In [12]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(df['genres'])

genre_df = pd.DataFrame(
    genre_matrix,
    columns=mlb.classes_,
    index=df.index
)

df = pd.concat([df, genre_df], axis=1)

Save genre names for later use:

In [13]:
import pickle
with open('../data/genre_columns.pkl', 'wb') as f:
    pickle.dump(mlb.classes_, f)

Drop uneccesary columns:

In [14]:
cols_to_drop = [
    'detailed_description',
    'about_the_game',
    'reviews',
    'header_image',
    'tags', 
]

df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

Export cleaned data into pickle file to maintain data types for model use, and csv to read independently.

In [15]:
df.to_pickle('../data/cleaned_games.pkl')
df.to_csv('../data/cleaned_games.csv', index=False)